**House Prices - Model Experiments**

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


*Data Loading* 

In [3]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print(train.shape)
print(test.shape)

print(train.head())

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [5]:
train.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

*Data Cleaning*

In [6]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)

In [7]:
test_ids = X["Id"]
X = X.drop("Id", axis=1)
test = test.drop("Id", axis=1)


In [8]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(test[num_cols].median())

In [9]:
cat_cols = X.select_dtypes(include=["object"]).columns
X[cat_cols] = X[cat_cols].fillna("None")
test[cat_cols] = test[cat_cols].fillna("None")

*Feature Engineering*

In [10]:
X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
test["TotalSF"] = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]

X["Age"] = X["YrSold"] - X["YearBuilt"]
test["Age"] = test["YrSold"] - test["YearBuilt"]

X["TotalBath"] = (
    X["FullBath"] +
    0.5 * X["HalfBath"] +
    X["BsmtFullBath"] +
    0.5 * X["BsmtHalfBath"]
)

test["TotalBath"] = (
    test["FullBath"] +
    0.5 * test["HalfBath"] +
    test["BsmtFullBath"] +
    0.5 * test["BsmtHalfBath"]
)

X["TotalPorch"] = (
    X["OpenPorchSF"] +
    X["EnclosedPorch"] +
    X["3SsnPorch"] +
    X["ScreenPorch"]
)

test["TotalPorch"] = (
    test["OpenPorchSF"] +
    test["EnclosedPorch"] +
    test["3SsnPorch"] +
    test["ScreenPorch"]
)

X["HasGarage"] = (X["GarageArea"] > 0).astype(int)
test["HasGarage"] = (test["GarageArea"] > 0).astype(int)

X["HasBasement"] = (X["TotalBsmtSF"] > 0).astype(int)
test["HasBasement"] = (test["TotalBsmtSF"] > 0).astype(int)

In [13]:
skewed = ["GrLivArea", "LotArea"]

for col in skewed:
    X[col] = np.log1p(X[col])
    test[col] = np.log1p(test[col])


y = np.log1p(y)

*Feature Selection*

In [11]:
outliers = (X["GrLivArea"] > 4000)
X = X[~outliers]
y = y[~outliers]

*Encoding*

In [12]:
X = pd.get_dummies(X)
test = pd.get_dummies(test)

X, test = X.align(test, join='left', axis=1, fill_value=0)

*Test Train Split*

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

*Mlflow Setup*

In [15]:
!pip install mlflow dagshub

In [16]:
import mlflow
import dagshub

dagshub.init(repo_owner='slosa23', repo_name='ML-Assignment1', mlflow=True)

Accessing as slosa23

Initialized MLflow to track repo "slosa23/ML-Assignment1"

Repository slosa23/ML-Assignment1 initialized!

*Model Training - Linear Regression*

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_name="linear-regression"):

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(model, "model")

    print("RMSE:", rmse)

*Model Training - Random Forest*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name="random-forest"):

    model = RandomForestRegressor(n_estimators=200, max_depth=10)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(model, "model")

    print("RMSE:", rmse)

*Potential Overfit*

In [ ]:
with mlflow.start_run(run_name="potential-overfit"):

    model = RandomForestRegressor(n_estimators=500, max_depth=None)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RF_overfit")
    mlflow.log_metric("rmse", rmse)

    print("RMSE:", rmse)

*Potential Underfit*

In [ ]:
with mlflow.start_run(run_name="potential-underfit"):

    model = RandomForestRegressor(n_estimators=10, max_depth=2)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RF_underfit")
    mlflow.log_metric("rmse", rmse)

    print("RMSE:", rmse)

*Hyperparameter Tuning - RF*

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    "n_estimators" : [100, 200, 300, 500],
    "max_depth" : [None, 5, 10, 20],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4]
}

rf = RandomForestRegressor(random_state = 42)

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

with mlflow.start_run(run_name="rf_tuned"):

    preds = best_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_params(search.best_params_)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(best_model, "model")

    print("Best params:", search.best_params_)
    print("RMSE:", rmse)

*Gradient Boosting*

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

with mlflow.start_run(run_name="gradient-boosting"):

    model = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "GradientBoosting")
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(model, "model")

    print("RMSE:", rmse)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

with mlflow.start_run(run_name="hist_gb"):

    gb_model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=10,
        max_iter=500,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    preds = gb_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "HistGradientBoostingRegressor")
    mlflow.log_params({
        "learning_rate": 0.05,
        "max_depth": 10,
        "max_iter": 500
    })
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(gb_model, "model")

    print("GB RMSE:", rmse)

In [ ]:
from sklearn.linear_model import Ridge

with mlflow.start_run(run_name="ridge"):

    ridge_model = Ridge(alpha=10)

    ridge_model.fit(X_train, y_train)

    preds = ridge_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 10)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(ridge_model, "model")

    print("Ridge RMSE:", rmse)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {"alpha": [0.1, 1, 5, 10, 20, 50]}

grid = GridSearchCV(
    Ridge(),
    params,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

grid.fit(X_train, y_train)

print(grid.best_params_)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

params = {
    "ridge__alpha": [0.01, 0.1, 1, 5, 10, 20, 50, 100, 200]
}

grid = GridSearchCV(
    pipeline,
    params,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'ridge__alpha': [0.01, 0.1, 1, 5, 10, 20, 50, 100,
                                          200]},
             scoring='neg_root_mean_squared_error')

In [21]:
from sklearn.metrics import mean_squared_error

best_model = grid.best_estimator_

with mlflow.start_run(run_name="ridge_model"):

    preds = best_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    # Access alpha from the Ridge step in the pipeline
    mlflow.log_params({
        "model": "Ridge Pipeline",
        "alpha": best_model.named_steps["ridge"].alpha
    })

    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(best_model, "model")

    print("Final Ridge RMSE:", rmse)

2026/04/11 19:27:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 19:27:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final Ridge RMSE: 0.11948904964313452
🏃 View run ridge_model at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0/runs/0e221d4a41524bcaa6b6571a08cba4bb
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0


In [22]:
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn
import numpy as np

best_model = grid.best_estimator_

with mlflow.start_run(run_name="ridge_final_model"):

    # --- Evaluate on validation ---
    val_preds = best_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))

    # --- Retrain on FULL data ---
    best_model.fit(X, y)

    # --- Log params ---
    mlflow.log_params({
        "model": "Ridge Pipeline",
        "alpha": best_model.named_steps["ridge"].alpha
    })

    # --- Log metric ---
    mlflow.log_metric("rmse", rmse)

    # --- Log FULL trained model ---
    mlflow.sklearn.log_model(best_model, "model")

    print("Final Ridge RMSE (val):", rmse)

2026/04/11 19:36:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 19:36:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final Ridge RMSE (val): 0.11948904964313452
🏃 View run ridge_final_model at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0/runs/0b41e892ec354a70956eb611bf1858bf
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0
